# Dataset 与 DataLoader 练习

本模块练习数据集协议、批处理、打乱、可复现划分和变长数据整理。所有数据均在本地生成。

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(42)

## 练习 1：TensorDataset ⭐

把 features 和 labels 封装成 `TensorDataset`，取出索引 2 的样本。

In [4]:
features = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
labels = torch.tensor([0, 1, 0, 1])

# TODO
dataset = TensorDataset(features,labels)
sample_feature, sample_label = dataset[2]

assert len(dataset) == 4
assert torch.equal(sample_feature, torch.tensor([5.0, 6.0]))
assert sample_label.item() == 0
print("✅ 练习 1 通过")

✅ 练习 1 通过


## 练习 2：自定义 Dataset ⭐⭐

补全 Dataset 类的 `__len__` 和 `__getitem__`。每次返回一个特征 Tensor 和对应标签。

In [5]:
class PairDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        # TODO
        return len(self.features)

    def __getitem__(self, index):
        # TODO
        return self.features[index],self.labels[index]


custom_dataset = PairDataset(features, labels)
f, label = custom_dataset[1]
assert len(custom_dataset) == 4
assert torch.equal(f, torch.tensor([3.0, 4.0]))
assert label.item() == 1
print("✅ 练习 2 通过")

✅ 练习 2 通过


## 练习 3：按批加载 ⭐⭐

创建批大小为 3、不打乱数据的 DataLoader，取出前两个批次并观察最后一个批次的大小。

In [10]:
# TODO
loader = DataLoader(dataset=dataset,batch_size=3,shuffle=False)
print(list(loader),'\n',list(loader.dataset))
batches = list(loader)[:2]

assert len(batches) == 2
assert batches[0][0].shape == (3, 2)
assert batches[0][1].shape == (3,)
assert batches[1][0].shape == (1, 2)
assert torch.equal(batches[0][1], torch.tensor([0, 1, 0]))
print("✅ 练习 3 通过")

[[tensor([[1., 2.],
        [3., 4.],
        [5., 6.]]), tensor([0, 1, 0])], [tensor([[7., 8.]]), tensor([1])]] 
 [(tensor([1., 2.]), tensor(0)), (tensor([3., 4.]), tensor(1)), (tensor([5., 6.]), tensor(0)), (tensor([7., 8.]), tensor(1))]
✅ 练习 3 通过


## 练习 4：可复现的数据集划分 ⭐⭐

用比例 75%/25% 划分一个包含 20 个样本的数据集。使用种子为 123 的独立 Generator，使重复划分得到相同索引。

In [15]:
full_dataset = TensorDataset(torch.arange(20))

# TODO
# 分别创建两个种子相同的 Generator，并执行两次 random_split
Generator1 = torch.Generator().manual_seed(123)
allen = len(full_dataset)
rt_train = 0.75
rt_val = 0.25
train_len = int(rt_train*allen)
val_len = int(rt_val*allen)
train_set_1, val_set_1 = random_split(
    dataset = full_dataset,
    lengths = [train_len,val_len],
    generator = Generator1
)
print(list(train_set_1),'\n',list(val_set_1))
Generator2 = torch.Generator().manual_seed(123)
train_set_2, val_set_2 = random_split(
    dataset=full_dataset,
    lengths=[train_len,val_len],
    generator=Generator2
)

assert len(train_set_1) == 15 and len(val_set_1) == 5
assert train_set_1.indices == train_set_2.indices
assert val_set_1.indices == val_set_2.indices
print("✅ 练习 4 通过")

[(tensor(2),), (tensor(9),), (tensor(14),), (tensor(15),), (tensor(16),), (tensor(7),), (tensor(8),), (tensor(5),), (tensor(19),), (tensor(0),), (tensor(11),), (tensor(18),), (tensor(10),), (tensor(17),), (tensor(12),)] 
 [(tensor(6),), (tensor(1),), (tensor(13),), (tensor(3),), (tensor(4),)]
✅ 练习 4 通过


## 练习 5：可复现的 shuffle ⭐⭐⭐

创建两个使用相同随机种子的 DataLoader。批大小为 5，并启用 shuffle。取出它们第一轮的完整样本顺序。

In [19]:
number_dataset = TensorDataset(torch.arange(20))

# TODO
Generator1 = torch.Generator().manual_seed(42)
Generator2 = torch.Generator().manual_seed(42)
loader_1 = DataLoader(dataset=number_dataset,batch_size=5,shuffle=True,generator=Generator1)
loader_2 = DataLoader(dataset=number_dataset,batch_size=5,shuffle=True,generator=Generator2)
order_1 = torch.tensor(list(loader_1.sampler))
order_2 = torch.tensor(list(loader_2.sampler))

assert order_1.shape == (20,)
assert torch.equal(order_1, order_2)
assert not torch.equal(order_1, torch.arange(20))
print("✅ 练习 5 通过")

✅ 练习 5 通过


## 练习 6：为变长序列编写 collate_fn ⭐⭐⭐

每个样本是一条长度不同的整数序列。补全 `pad_collate`，将一个批次填充成二维 Tensor，填充值为 0，同时返回原始长度。

In [22]:
sequences = [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6])]

class SequenceDataset(Dataset):
    def __len__(self):
        return len(sequences)

    def __getitem__(self, index):
        return sequences[index]


def pad_collate(batch):
    # TODO
    lengths = torch.tensor([len(bh) for bh in batch])
    max_len = max(lengths)
    padded = torch.zeros(max_len,max_len)
    for i,bh in enumerate(batch):
        padded[i,:len(bh)] = batch[i]
    return padded,lengths

sequence_loader = DataLoader(SequenceDataset(), batch_size=3, collate_fn=pad_collate)
padded, lengths = next(iter(sequence_loader))
print(padded,lengths)
assert torch.equal(padded, torch.tensor([[1, 2, 3], [4, 5, 0], [6, 0, 0]]))
assert torch.equal(lengths, torch.tensor([3, 2, 1]))
print("✅ 练习 6 通过")

tensor([[1., 2., 3.],
        [4., 5., 0.],
        [6., 0., 0.]]) tensor([3, 2, 1])
✅ 练习 6 通过


## 过关标准

你应该能说明 Dataset 负责什么、DataLoader 负责什么，以及 batch size、shuffle、drop_last、generator 和 collate_fn 分别影响哪一环。  
答：Dataset负责将数据整合成一个数据集合，DataLoader负责按照sampler的顺序，把数据按批次输出，batch size是批大小，shuffle选择是否  
打乱顺序，drop_last选择是否丢弃最后一个批次，generator选择随机数种子，collate_fn实现每个批次的数据处理和返回。